In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Chi-squared-example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [4]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('D:\my-repo\data\students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

In [6]:
from pyspark.ml.feature import VectorAssembler, ChiSqSelector

# Подготовка на данните
feature_cols = ["Age","PlatformHours","OutPlatformHours","Self-assessment"]
# Обединяване на характеристиките във вектор
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features")
data = assembler.transform(df)

# Селектиране на 2 най-информативни характеристики
selector = ChiSqSelector(numTopFeatures=2, featuresCol="features",
    outputCol="selectedFeatures", labelCol="Satisfaction")
# Обучение на модела
model = selector.fit(data)
# Трансформация на данните
result = model.transform(data)

# Индекси на избраните характеристики
selected_indices = model.selectedFeatures

print("Избрани характеристики:")
for index in selected_indices:
    print(feature_cols[index])

# Показване на избраните характеристики
result.select("features", "selectedFeatures").show(truncate=False)    

Избрани характеристики:
OutPlatformHours
Self-assessment
+-------------------+----------------+
|features           |selectedFeatures|
+-------------------+----------------+
|[21.0,6.0,2.0,4.0] |[2.0,4.0]       |
|[23.0,8.0,1.0,5.0] |[1.0,5.0]       |
|[20.0,0.0,3.0,3.0] |[3.0,3.0]       |
|[22.0,5.0,1.0,4.0] |[1.0,4.0]       |
|[24.0,9.0,0.0,5.0] |[0.0,5.0]       |
|[21.0,0.0,2.0,3.0] |[2.0,3.0]       |
|[22.0,7.0,1.0,4.0] |[1.0,4.0]       |
|[23.0,6.0,0.0,4.0] |[0.0,4.0]       |
|[20.0,0.0,2.0,2.0] |[2.0,2.0]       |
|[25.0,10.0,0.0,5.0]|[0.0,5.0]       |
|[21.0,4.0,1.0,3.0] |[1.0,3.0]       |
|[22.0,7.0,0.0,4.0] |[0.0,4.0]       |
+-------------------+----------------+

